# Spectroscopy workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

> [!WARNING]
> Operations in this notebook change control-instrument settings, including LO and NCO values. Running them can affect other users sharing the same hardware, so use this notebook only when that impact is acceptable.

This notebook walks through a full spectroscopy bring-up sequence for one multiplexed readout group: locate the resonators, refine their readout frequencies, identify the qubit transitions, and finish by updating the default control amplitudes. It assumes you will inspect the plots, edit the configuration files manually where noted, and reload before continuing.

## 1. Create an `Experiment`

Start from the mux you want to characterize, then load the corresponding configuration and parameter files for your system.

In [1]:
import numpy as np

import qubex_emulator as qx

exp = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connecting also synchronizes the clocks, so do this before running any spectroscopy measurement.

In [2]:
# Clocks will be synced when connected
exp.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Run a coarse resonator scan

Begin with a broad readout-frequency sweep on one representative qubit to identify the frequency region that covers all resonators in the mux. Replace the placeholder range below with one wide enough to include every expected resonator frequency in your device.

In [3]:
# Sweep readout frequency to locate the resonator band
# Set the readout amplitude to a reasonable value
exp.scan_resonator_frequencies(
    exp.qubit_labels[0],  # Qubit label for the associated resonator
    frequency_range=np.arange(9.75, 10.75, 0.002),  # Placeholder example in GHz
    readout_amplitude=0.1,
    save_image=True,
)

{'frequency_range': array([ 9.75 ,  9.752,  9.754,  9.756,  9.758,  9.76 ,  9.762,  9.764,
         9.766,  9.768,  9.77 ,  9.772,  9.774,  9.776,  9.778,  9.78 ,
         9.782,  9.784,  9.786,  9.788,  9.79 ,  9.792,  9.794,  9.796,
         9.798,  9.8  ,  9.802,  9.804,  9.806,  9.808,  9.81 ,  9.812,
         9.814,  9.816,  9.818,  9.82 ,  9.822,  9.824,  9.826,  9.828,
         9.83 ,  9.832,  9.834,  9.836,  9.838,  9.84 ,  9.842,  9.844,
         9.846,  9.848,  9.85 ,  9.852,  9.854,  9.856,  9.858,  9.86 ,
         9.862,  9.864,  9.866,  9.868,  9.87 ,  9.872,  9.874,  9.876,
         9.878,  9.88 ,  9.882,  9.884,  9.886,  9.888,  9.89 ,  9.892,
         9.894,  9.896,  9.898,  9.9  ,  9.902,  9.904,  9.906,  9.908,
         9.91 ,  9.912,  9.914,  9.916,  9.918,  9.92 ,  9.922,  9.924,
         9.926,  9.928,  9.93 ,  9.932,  9.934,  9.936,  9.938,  9.94 ,
         9.942,  9.944,  9.946,  9.948,  9.95 ,  9.952,  9.954,  9.956,
         9.958,  9.96 ,  9.962,  9.964,  9.96

## 4. Sweep readout frequency and power

Use the coarse scan to define a narrower window, then build a 2D resonator spectroscopy map to choose a practical readout power. Convert the selected power into the amplitude used by the lower-level scan APIs.

In [4]:
# Choose a frequency range that covers all resonator frequencies
readout_frequency_range = np.arange(10.05, 10.55, 0.002)

In [5]:
# Sweep readout frequency and power
exp.resonator_spectroscopy(
    exp.qubit_labels[0],
    frequency_range=readout_frequency_range,
    power_range=np.arange(-60, 5, 5),
)

{'frequency_range': array([10.05 , 10.052, 10.054, 10.056, 10.058, 10.06 , 10.062, 10.064,
        10.066, 10.068, 10.07 , 10.072, 10.074, 10.076, 10.078, 10.08 ,
        10.082, 10.084, 10.086, 10.088, 10.09 , 10.092, 10.094, 10.096,
        10.098, 10.1  , 10.102, 10.104, 10.106, 10.108, 10.11 , 10.112,
        10.114, 10.116, 10.118, 10.12 , 10.122, 10.124, 10.126, 10.128,
        10.13 , 10.132, 10.134, 10.136, 10.138, 10.14 , 10.142, 10.144,
        10.146, 10.148, 10.15 , 10.152, 10.154, 10.156, 10.158, 10.16 ,
        10.162, 10.164, 10.166, 10.168, 10.17 , 10.172, 10.174, 10.176,
        10.178, 10.18 , 10.182, 10.184, 10.186, 10.188, 10.19 , 10.192,
        10.194, 10.196, 10.198, 10.2  , 10.202, 10.204, 10.206, 10.208,
        10.21 , 10.212, 10.214, 10.216, 10.218, 10.22 , 10.222, 10.224,
        10.226, 10.228, 10.23 , 10.232, 10.234, 10.236, 10.238, 10.24 ,
        10.242, 10.244, 10.246, 10.248, 10.25 , 10.252, 10.254, 10.256,
        10.258, 10.26 , 10.262, 10.264, 10.26

In [6]:
# Choose a readout power and convert it to amplitude
readout_power = -40  # dB
readout_amplitude = 10 ** (readout_power / 20)
readout_amplitude

0.01

## 5. Re-scan the resonators and map the peaks

With the readout amplitude fixed, repeat the resonator scan, extract the fitted peaks, and assign those peak frequencies back to qubit labels in the chip-specific order for this mux.

In [7]:
# Re-scan readout frequency with the selected power
result = exp.scan_resonator_frequencies(
    exp.qubit_labels[0],
    frequency_range=readout_frequency_range,
    readout_amplitude=readout_amplitude,
    save_image=True,
)

# Extract the fitted peaks
peaks = result["peaks"]

In [8]:
# Map resonator frequencies to qubit labels
# The ordering depends on the chip design
resonator_frequencies = {
    exp.qubit_labels[0]: peaks[1],
    exp.qubit_labels[1]: peaks[3],
    exp.qubit_labels[2]: peaks[2],
    exp.qubit_labels[3]: peaks[0],
}
resonator_frequencies

{'Q00': 10.216000000000056,
 'Q01': 10.548000000000167,
 'Q02': 10.382000000000112,
 'Q03': 10.05}

## 6. Persist and refine the resonator frequencies

Write the coarse resonator frequencies to `readout_frequency.yaml`, reload, and then measure the reflection coefficient for each qubit to extract a more precise fitted resonator frequency. After updating `readout_frequency.yaml` again, reload once more before moving to qubit spectroscopy.

In [9]:
# Update `readout_frequency.yaml` manually, then reload
exp.reload()

In [10]:
# Fit the reflection coefficient at resonance
fine_resonator_frequencies = {}
for qubit in exp.qubit_labels:
    result = exp.measure_reflection_coefficient(
        qubit,
        readout_amplitude=readout_amplitude,
    )
    fine_resonator_frequencies[qubit] = result["f_r"]

fine_resonator_frequencies

{'Q00': 6.752, 'Q01': 6.903, 'Q02': 6.844, 'Q03': 6.711}

In [11]:
# Update `readout_frequency.yaml` again, then reload
exp.reload()

## 7. Find rough qubit frequencies

Run a first-pass control-frequency sweep for each qubit, then define a per-qubit frequency window that covers the `ge` and `ef` transitions you want to inspect more closely.

In [12]:
# Sweep control frequency to find the qubit frequency
# The qubit frequency is ac-Stark shifted by the photon number in the resonator,
# so keep `readout_amplitude` as low as possible while the signal remains visible
for qubit in exp.qubit_labels:
    exp.scan_qubit_frequencies(
        qubit,
        control_amplitude=0.1,
        readout_amplitude=0.01,
    )

In [13]:
# Choose a control-frequency range that covers the `ge` and `ef` transitions
control_frequency_ranges = {
    exp.qubit_labels[0]: np.arange(6.5, 7.5, 0.005),
    exp.qubit_labels[1]: np.arange(7.5, 8.5, 0.005),
    exp.qubit_labels[2]: np.arange(7.5, 8.5, 0.005),
    exp.qubit_labels[3]: np.arange(6.5, 7.7, 0.005),
}

## 8. Sweep qubit frequency and fit the resonance

Use a 2D qubit spectroscopy scan to choose the operating region, measure the resonance for each qubit, and then update `control_frequency.yaml` with the fitted `qubit_frequency` values before reloading.

In [14]:
# Sweep control frequency and power
# The qubit frequency is ac-Stark shifted by the photon number in the resonator,
# so keep `readout_amplitude` as low as possible while the signal remains visible
for qubit in exp.qubit_labels:
    exp.qubit_spectroscopy(
        qubit,
        frequency_range=control_frequency_ranges[qubit],
        power_range=np.arange(-60, 5, 5),
        readout_amplitude=0.01,
    )

In [15]:
# Fit the qubit frequency at resonance
for qubit in exp.qubit_labels:
    exp.measure_qubit_resonance(
        qubit,
        frequency_range=control_frequency_ranges[qubit],
        readout_amplitude=0.03,
    )

In [16]:
# Update `control_frequency.yaml` manually, then reload
exp.reload()

## 9. Validate the waveform and update control amplitudes

After the frequency updates, check the readout waveform, obtain a Rabi fit, and convert that result into default control amplitudes. Once you edit `control_amplitude.yaml`, reload so the final validation uses the new values.

In [17]:
# Check the readout-pulse waveform
exp.check_waveform()

{'waveform': None, 'ok': True}

In [18]:
# Measure the Rabi oscillation
exp.obtain_rabi_params()

In [19]:
# Calculate the default control amplitudes
exp.calc_control_amplitudes()

{'Q00': 1.0, 'Q01': 1.0, 'Q02': 1.0, 'Q03': 1.0}

In [20]:
# Update `control_amplitude.yaml` manually, then reload
exp.reload()

## 10. Re-check the Rabi oscillation

Run the Rabi measurement again to confirm the updated default amplitudes behave as expected.

In [21]:
# Re-check the Rabi oscillation
exp.obtain_rabi_params()